In [1]:
print("hi")

hi


In [1]:
import os
import re
import json
import pdfplumber
import traceback

from PIL import Image
from transformers import pipeline
from output_utils import save_split_output
from confidence_utils import (
    calculate_eob_confidence,
    _unwrap_vlm_output,calculate_model_confidence
)

pipe = pipeline(
    "image-text-to-text",
    model="Qwen/Qwen3-VL-2B-Instruct",
    device_map="auto"
)




COLUMN_HEADERS = [
    "procedure_code",
    "dos",
    "amount_claimed",
    "amount_allowed",
    "deduct_applied",
    "other_ins",
    "patient_resp",
    "amount_paid"
]

def build_prompt(expected_rows=None):

    row_instruction = ""

    if expected_rows is not None:
        row_instruction = f"""
SERVICE ROW COUNT:
- This table contains EXACTLY {expected_rows} service rows.
- You MUST return EXACTLY {expected_rows} rows in "rows" array.
- No more, no less.
- Count only lines that have ADA CODE + DATE OF SERVICE together.
- Description lines below ADA CODE are NOT rows — do not count them.
"""

    return f"""You are extracting highly sensitive dental EOB financial table data from a PDF image.

STRICT EXTRACTION RULES:

1. Extract ONLY table data rows.
2. Each row MUST have a valid ADA CODE on the SAME horizontal line.
3. NEVER extract a description line as a row.
4. NEVER merge rows.
5. NEVER infer values from nearby rows.
6. NEVER copy values from adjacent columns.
7. NEVER borrow values from neighboring tables.
8. Duplicate values are VALID financial data — keep all rows.
9. NEVER skip any row.
10. Preserve row order exactly as shown.
11. If a cell is empty, return "".
12. Extract values ONLY from their exact columns.

HOW TO IDENTIFY A VALID DATA ROW:

A valid data row has ALL of these on the SAME horizontal line:
- ADA CODE (e.g. "ADA CODE D2392")
- DATE OF SERVICE (e.g. "02/03/26")
- Monetary values

A DESCRIPTION LINE has only text — NO ADA CODE, NO DATE, NO monetary values.
→ DESCRIPTION LINES MUST BE COMPLETELY IGNORED. DO NOT EXTRACT THEM.

EXAMPLE — 2 rows with description lines:

  ADA CODE D4341   03/02/26  $367.00  $237.00  $0.00  $0.00  $118.50  $118.50  ← ROW 1
  periodontal scaling and root planing...                                        ← DESCRIPTION, SKIP
  ADA CODE D4341   03/02/26  $367.00  $237.00  $0.00  $0.00  $118.50  $118.50  ← ROW 2
  periodontal scaling and root planing...                                        ← DESCRIPTION, SKIP

CORRECT OUTPUT: 2 rows
WRONG OUTPUT:   4 rows (if description lines extracted as rows)

{row_instruction}

IMPORTANT — MULTIPLE TABLES IN ONE IMAGE:

- Image may contain more than one table stitched vertically.
- Extract rows from ALL tables in the image.
- Do NOT stop after the first table.
- Scan entire image top to bottom.

COLUMN MAPPING:

- ADA CODE -> procedure_code
- DATE OF SERVICE -> dos
- AMOUNT CLAIMED -> amount_claimed
- AMOUNT ALLOWED -> amount_allowed
- DEDUCT APPLIED -> deduct_applied
- OTHER INS -> other_ins
- PATIENT RESP -> patient_resp
- AMOUNT PAID -> amount_paid

IGNORE THESE COLUMNS COMPLETELY:
- TOOTH NO
- EOB CODE
- DESCRIPTION

PROCEDURE CODE RULES:

- Valid format: starts with "D" + exactly 4 digits.
- Same ADA CODE multiple times = multiple valid rows — extract ALL.
- ONE ADA CODE line = ONE ROW. Never split, never merge.

DATE RULE:
- Extract only DATE OF SERVICE from the same row line.

MONETARY FIELD RULES:
- Extract exact values only from the SAME horizontal line as the ADA CODE.
- Do NOT borrow values from above or below.
- Do NOT copy from a different table.

TOTAL ROW RULES:
1. SUB-TOTAL row = totals row → extract into "column_totals".
2. Use LAST SUB-TOTAL in image for "column_totals".
3. Do NOT include SUB-TOTAL inside normal rows.
4. Any row without both ADA CODE and DATE OF SERVICE → skip it.

OUTPUT FORMAT:

Return ONLY valid JSON.
No explanation.
No markdown.
No extra text.

IMPORTANT:
EVERY FIELD MUST BE RETURNED USING THIS FORMAT:

{{
    "value": "",
    "confidence": 0.0
}}

NEVER return a field as a plain string.

For example, DO NOT return:

"procedure_code": "D4341"

ALWAYS return:

"procedure_code": {{
    "value": "D4341",
    "confidence": 0.99
}}

Confidence rules:
- confidence must be a number between 0.0 and 1.0.
- 1.0 = completely certain that the value was correctly read.
- 0.0 = value is missing, unreadable, or cannot be reliably extracted.
- Do not guess values.
- If a value cannot be reliably extracted:
    "value": "",
    "confidence": 0.0
- Confidence represents confidence in reading the value from the image.
- Do not calculate confidence based on the financial amount itself.

EVERY field in EVERY row and EVERY field in column_totals MUST contain
both "value" and "confidence".

FINAL JSON STRUCTURE:

{{
    "rows": [
        {{
            "procedure_code": {{
                "value": "",
                "confidence": 0.0
            }},
            "dos": {{
                "value": "",
                "confidence": 0.0
            }},
            "amount_claimed": {{
                "value": "",
                "confidence": 0.0
            }},
            "amount_allowed": {{
                "value": "",
                "confidence": 0.0
            }},
            "deduct_applied": {{
                "value": "",
                "confidence": 0.0
            }},
            "other_ins": {{
                "value": "",
                "confidence": 0.0
            }},
            "patient_resp": {{
                "value": "",
                "confidence": 0.0
            }},
            "amount_paid": {{
                "value": "",
                "confidence": 0.0
            }}
        }}
    ],

    "column_totals": {{
        "amount_claimed": {{
            "value": "",
            "confidence": 0.0
        }},
        "amount_allowed": {{
            "value": "",
            "confidence": 0.0
        }},
        "deduct_applied": {{
            "value": "",
            "confidence": 0.0
        }},
        "other_ins": {{
            "value": "",
            "confidence": 0.0
        }},
        "patient_resp": {{
            "value": "",
            "confidence": 0.0
        }},
        "amount_paid": {{
            "value": "",
            "confidence": 0.0
        }}
    }}

}}

FINAL CHECK BEFORE RESPONDING:

Verify that EVERY field has:
- "value"
- "confidence"

If any field is a plain string, convert it to the required
"value + confidence" structure before returning the JSON.

Return ONLY JSON.
"""
# =========================================================
# IMAGE -> JSON EXTRACTION
# =========================================================

def extract_table_from_image(image_path, expected_rows=None):

    image = Image.open(image_path).convert("RGB")
    dynamic_prompt = build_prompt(expected_rows)

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image
                },
                {
                    "type": "text",
                    "text": dynamic_prompt
                }
            ]
        }
    ]

    output = pipe(
        text=messages,
        max_new_tokens=2048
    )

    generated_text = output[0]["generated_text"]

    if isinstance(generated_text, list):
        generated_text = generated_text[-1]["content"]

    generated_text = generated_text.strip()

    generated_text = generated_text.replace(
        "```json", ""
    ).replace(
        "```", ""
    ).strip()

    return generated_text


# =========================================================
# ENFORCE SCHEMA
# =========================================================

def enforce_schema(parsed_output):
    allowed_columns = COLUMN_HEADERS

    for table in parsed_output.get("tables", []):
        cleaned_rows = []
        detected_totals = {}

        for row in table.get("rows", []):
            cleaned_row = {}

            for col in allowed_columns:
                value = row.get(col, "")
                if value is None:
                    value = ""
                value = str(value).strip()

                if col == "dos":
                    date_match = re.search(r"\d{2}/\d{2}/\d{2,4}", value)
                    value = date_match.group(0) if date_match else ""

                elif col == "procedure_code":

                    match = re.search(
                        r"\b(D\d{4})\b",
                        value,
                        re.IGNORECASE
                    )

                    value = match.group(1).upper() if match else ""

                else:
                    money_match = re.search(r"\$?([\d,]+\.\d{2})", value)
                    value = money_match.group(1) if money_match else ""

                cleaned_row[col] = value

            is_total_row = (
                cleaned_row["dos"] == ""
                and cleaned_row["procedure_code"] == ""
            )

            if is_total_row:
                for col in allowed_columns:
                    if col not in ["dos", "procedure_code"]:
                        val = cleaned_row.get(col, "")
                        if val != "":
                            detected_totals[col] = val
                continue

            valid_code = re.fullmatch(
                r"D\d{4}",
                cleaned_row["procedure_code"]
            )

            valid_date = re.fullmatch(
                r"\d{2}/\d{2}/\d{2,4}",
                cleaned_row["dos"]
            )

            if valid_code and valid_date:
                cleaned_rows.append(cleaned_row)

        table["rows"] = cleaned_rows

        final_totals = {}
        existing_totals = table.get("column_totals", {})

        for col in allowed_columns:
            if col not in ["dos", "procedure_code"]:
                final_totals[col] = detected_totals.get(
                    col, existing_totals.get(col, "")
                )

        table["column_totals"] = final_totals

    return parsed_output


# =========================================================
# FINANCIAL TOTAL VALIDATION
# =========================================================

def validate_eob_table(table, t_idx):
    money_cols = [
        "amount_claimed", "amount_allowed", "deduct_applied",
        "other_ins", "patient_resp", "amount_paid"
    ]

    rows = table.get("rows", [])
    column_totals = table.get("column_totals", {})

    print(f"\n🔍 Validation for [Table {t_idx}]")
    print("-" * 75)

    all_passed = True
    errors = []

    for col in money_cols:
        computed = 0.0
        for row in rows:
            val = row.get(col, "").replace("$", "").replace(",", "").strip()
            try:
                computed += float(val)
            except ValueError:
                pass

        extracted_str = column_totals.get(col, "").replace("$", "").replace(",", "").strip()

        try:
            extracted = float(extracted_str)
        except ValueError:
            extracted = None

        if extracted is None:
            status, label = "⚠️ ", "missing"
            all_passed = False
            errors.append({"field": col, "computed": computed, "extracted": None, "type": "missing"})
        elif abs(computed - extracted) < 0.02:
            status, label = "✅", "match"
        else:
            status, label = "❌", "MISMATCH"
            all_passed = False
            errors.append({"field": col, "computed": computed, "extracted": extracted, "type": "field_mismatch"})

        print(f"{status} {col:<28}computed={computed:<12.1f}| extracted={extracted if extracted is not None else 'N/A':<12}{label}")

    print("-" * 75)
    if all_passed:
        print(f"✅ [Table {t_idx}] Validation PASSED")
    else:
        print(f"❌ [Table {t_idx}] Validation FAILED")

    return all_passed, errors, len(money_cols)  # ✅ now returns something usable

# =========================================================
# DENIAL DETECTION IN A REGION
# =========================================================

def check_denial_in_region(page, start_y, end_y):
    region = page.crop((0, start_y, page.width, end_y))
    text = region.extract_text() or ""
    return bool(re.search(r"\b(denied|denial)\b", text, re.IGNORECASE))


# =========================================================
# PER-PAGE DENIAL SCAN (full page, no region restriction)
# Returns list of (y_position, matched_word) for every hit
# =========================================================

def find_denial_hits_on_page(page):
    """
    Returns a list of dicts with keys: top, bottom, text
    for every word on the page that matches 'denied' or 'denial'.
    """
    hits = []
    words = page.extract_words() or []
    for w in words:
        if re.search(r"\b(denied|denial)\b", w.get("text", ""), re.IGNORECASE):
            hits.append({
                "top":    w["top"],
                "bottom": w["bottom"],
                "text":   w["text"]
            })
    return hits


def extract_provider_name(page, start_y, end_y):
    """
    Extracts the provider (dentist/practice) name — the first line in
    the member block, printed directly above the 'NPI Submitted:' line.
    """
    try:
        # start higher up so the provider name line isn't cropped out
        member_region = page.crop((0, start_y, 320, start_y + 170))
        member_text = member_region.extract_text() or ""
        lines = [l.strip() for l in member_text.splitlines() if l.strip()]

        for i, line in enumerate(lines):
            if "NPI" in line.upper():
                if i - 1 >= 0:
                    candidate = lines[i - 1].strip()
                    # guard against grabbing a header/label line by mistake
                    if candidate and "NPI" not in candidate.upper():
                        return candidate
                break

        if lines:
            return lines[0]

    except Exception as e:
        print(f"  ❌ Provider extraction error: {e}")

    return ""

# =========================================================
# MAIN PIPELINE
# =========================================================

def crop_all_eob_tables(pdf_path, output_dir="EOB_OUTPUT/UHC_commercial", company_name = "UHC commercial"):
    pdf_dir = os.path.basename(pdf_path).split(".")[0].split("_")[-1]
    pdf_full_name = os.path.basename(pdf_path)

    output_dir = os.path.join(output_dir, pdf_dir)
    os.makedirs(output_dir, exist_ok=True)

    

    all_patients = []
    confidence_results = []

    global_table_idx = 0
    claim_denied = False  # claim-level flag: True if ANY logical table was denied

    # =========================================================
    # PASS 1: Scan all pages, collect all segments
    # =========================================================
    with pdfplumber.open(pdf_path) as pdf:

        all_segments = []

        for page_num, page in enumerate(pdf.pages, start=1):

            print(f"\nScanning Page {page_num}")

            # Per-page denial scan
            denial_hits = find_denial_hits_on_page(page)
            page_denied = bool(denial_hits)
            print(f"  Page {page_num} — denial keyword found: {page_denied}")

            # Find starts: "PROVIDER OR MBR NAME"
            start_hits = page.search("PROVIDER OR MBR NAME", case=False)
            if not start_hits:
                start_hits = page.search("MBR", case=False)

            # Find ends: "SUB-TOTAL"
            end_hits = page.search("SUB-TOTAL", case=False)

            provider_tops = sorted([
                max(0, h["top"] - 3) for h in start_hits
            ])
            subtotal_bottoms = sorted([
                h["bottom"] + 3 for h in end_hits
            ])

            print(f"  provider_tops   = {[f'{y:.1f}' for y in provider_tops]}")
            print(f"  subtotal_bottoms= {[f'{y:.1f}' for y in subtotal_bottoms]}")

            used_subtotals = set()

            for p_idx, p_top in enumerate(provider_tops):

                # Next provider start = upper boundary for subtotal search
                if p_idx + 1 < len(provider_tops):
                    next_provider_y = provider_tops[p_idx + 1]
                else:
                    next_provider_y = None

                # Find first SUB-TOTAL below this provider
                # AND above next provider (if exists)
                paired_subtotal = None
                for s_idx, sub_y in enumerate(subtotal_bottoms):
                    if s_idx not in used_subtotals and sub_y > p_top:
                        if next_provider_y is None or sub_y <= next_provider_y:
                            paired_subtotal = sub_y
                            used_subtotals.add(s_idx)
                            break

                if paired_subtotal is not None:
                    # ✅ Complete — ended by SUB-TOTAL
                    end_y = paired_subtotal
                    has_end = True
                    print(f"  ✅ Provider@{p_top:.1f} → SUB-TOTAL@{end_y:.1f}")

                elif next_provider_y is not None:
                    # ✅ Ended by next Provider Name on same page
                    end_y = next_provider_y
                    has_end = True
                    print(f"  ✅ Provider@{p_top:.1f} → next Provider@{end_y:.1f} (no SUB-TOTAL)")

                else:
                    # ⏳ No end found — continues to next page
                    end_y = page.height
                    has_end = False
                    print(f"  ⏳ Provider@{p_top:.1f} → page end (CONTINUING)")

                seg = {
                    "page_num": page_num,
                    "page_obj": page,
                    "start_y":  p_top,
                    "end_y":    end_y,
                    "has_end":  has_end,
                }
                all_segments.append(seg)

            # Unmatched SUB-TOTALs = continuation from previous page
            unmatched_subtotals = [
                subtotal_bottoms[i]
                for i in range(len(subtotal_bottoms))
                if i not in used_subtotals
            ]

            # Only add continuation if there's a pending segment
            has_pending = any(
                not s.get("has_end", True) and not s.get("is_continuation", False)
                for s in all_segments
            )

            for sub_y in unmatched_subtotals:
                if not has_pending:
                    print(f"  ⚠️ Skipping orphan SUB-TOTAL at y={sub_y:.1f}")
                    continue
                seg = {
                    "page_num":        page_num,
                    "page_obj":        page,
                    "start_y":         0,
                    "end_y":           sub_y,
                    "has_end":         True,
                    "is_continuation": True,
                }
                all_segments.append(seg)
                print(f"  🔗 Continuation segment on page {page_num}: end_y={sub_y:.1f}")

        # =========================================================
        # PASS 2: Group segments into logical tables
        # =========================================================
        logical_tables = []
        pending = []

        all_segments.sort(key=lambda s: (s["page_num"], s["start_y"]))

        for seg in all_segments:

            is_continuation = seg.get("is_continuation", False)

            if is_continuation:
                if pending:
                    pending.append(seg)
                    logical_tables.append(list(pending))
                    pending = []
                else:
                    logical_tables.append([seg])

            elif not seg["has_end"]:
                if pending:
                    logical_tables.append(list(pending))
                    pending = []
                pending.append(seg)

            else:
                if pending:
                    # Same provider check → merge if same
                    pending_name = extract_provider_name_at_y(
                        pending[0]["page_obj"],
                        pending[0]["start_y"],
                        pending[0]["end_y"]
                    )
                    current_name = extract_provider_name_at_y(
                        seg["page_obj"],
                        seg["start_y"],
                        seg["end_y"]
                    )
                    if (
                        pending_name
                        and current_name
                        and pending_name.strip() == current_name.strip()
                    ):
                        pending.append(seg)
                        logical_tables.append(list(pending))
                        pending = []
                        print(f"  🔗 Merged same provider: '{current_name}'")
                    else:
                        logical_tables.append(list(pending))
                        pending = []
                        logical_tables.append([seg])
                else:
                    logical_tables.append([seg])

        if pending:
            logical_tables.append(list(pending))

        print(f"\nTotal logical tables detected: {len(logical_tables)}")

        # =========================================================
        # PASS 3: Crop + stitch + extract
        # =========================================================
        for lt_idx, segments in enumerate(logical_tables, start=1):

            print(f"\n--- Logical Table {lt_idx} ({len(segments)} segment(s)) ---")

            cropped_images = []

            for seg in segments:
                page     = seg["page_obj"]
                start_y  = seg["start_y"]
                end_y    = seg["end_y"]
                page_num = seg["page_num"]

                if start_y >= end_y:
                    print(f"  Skipping invalid bbox on page {page_num}")
                    continue

                bbox    = (0, start_y, page.width, end_y)
                cropped = page.crop(bbox)
                pil_img = cropped.to_image(resolution=300).original
                cropped_images.append((page_num, pil_img))
                print(f"  Cropped page {page_num}: y={start_y:.1f}→{end_y:.1f}")

            if not cropped_images:
                print(f"  No valid crops, skipping.")
                continue

            # Stitch
            if len(cropped_images) == 1:
                stitched = cropped_images[0][1]
            else:
                total_width  = max(img.width  for _, img in cropped_images)
                total_height = sum(img.height for _, img in cropped_images)
                stitched = Image.new("RGB", (total_width, total_height), color=(255, 255, 255))
                y_offset = 0
                for _, img in cropped_images:
                    stitched.paste(img, (0, y_offset))
                    y_offset += img.height

            global_table_idx += 1
            image_path = os.path.join(output_dir, f"table_{global_table_idx}.png")
            stitched.save(image_path)
            print(f"  Saved stitched image → {image_path}")

            first_seg = segments[0]

            # Denial check across all segments (this logical table only)
            table_denied = False
            for seg in segments:
                if check_denial_in_region(
                    seg["page_obj"],
                    seg["start_y"],
                    seg["end_y"]
                ):
                    table_denied = True
                    break

            # Roll this table's denial status up into the claim-level flag
            if table_denied:
                claim_denied = True

            print(f"  {'🔴 DENIED' if table_denied else '🟢 NOT DENIED'}")

            expected_rows = 0
            for seg in segments:
                ada_hits = seg["page_obj"].search("ADA CODE D", case=False)
                for hit in ada_hits:
                    hit_y = float(hit["top"])
                    if seg["start_y"] <= hit_y <= seg["end_y"]:
                        expected_rows += 1
            print (f"Expected rows: {expected_rows}")
            try:
                llm_output = extract_table_from_image(image_path, expected_rows= expected_rows)
                llm_output = (
                    llm_output
                    .replace("```json", "")
                    .replace("```", "")
                    .strip()
                )
                parsed_output = json.loads(llm_output)

                if "tables" not in parsed_output:
                    parsed_output = {"tables": [parsed_output]}

                # =========================================================
                # CONFIDENCE CALCULATION + UNWRAP
                # =========================================================

                for i, table in enumerate(parsed_output.get("tables", [])):

                    # Calculate confidence while the
                    # value + confidence structure still exists
                    table_model_confidence = calculate_model_confidence(table)

                    # Convert:
                    # {"value": "D4341", "confidence": 0.98}
                    #
                    # into:
                    # "D4341"
                    table = _unwrap_vlm_output(table)

                    # Keep the calculated confidence internally
                    table["_model_confidence"] = table_model_confidence

                    # Put the unwrapped table back
                    parsed_output["tables"][i] = table

                    # Save for final EOB confidence calculation
                    


                # =========================================================
                # NOW RUN YOUR EXISTING SCHEMA CLEANING
                # =========================================================


                parsed_output = enforce_schema(parsed_output)
                for table in parsed_output.get("tables", []):

                    rows = table.get("rows", [])

                    # Remove rows without ADA code
                    rows = [
                        r for r in rows
                        if re.fullmatch(
                            r"D\d{4}",
                            r.get("procedure_code", "")
                        )
                    ]

                    # Force exact count
                    if len(rows) > expected_rows:

                        print(
                            f"⚠️ Extra rows detected: "
                            f"{len(rows)} -> {expected_rows}"
                        )

                        rows = rows[:expected_rows]

                    table["rows"] = rows

                is_valid = True
                validation_errors = []

                for t_idx, table in enumerate(parsed_output.get("tables", []), start=1):
                    extracted_count = len(table.get("rows", []))
                    row_count_ok = (expected_rows == extracted_count)
                    if not row_count_ok:
                        is_valid = False
                        validation_errors.append({
                            "type": "row_count_mismatch",
                            "expected_rows": expected_rows,
                            "extracted_rows": extracted_count
                        })

                    table_valid, table_errors, total_fields = validate_eob_table(table, t_idx)
                    if not table_valid:
                        is_valid = False
                        validation_errors.extend(table_errors)

                # for t_idx, table in enumerate(
                #     parsed_output.get("tables", []), start=1
                # ):
                #     validate_eob_table(table, t_idx)

                for table in parsed_output.get("tables", []):

                    # Patient name extraction (your existing logic)
                    patient_name = ""
                    try:
                        member_region = first_seg["page_obj"].crop((
                            0,
                            first_seg["start_y"] + 70,
                            320,
                            first_seg["start_y"] + 170
                        ))
                        member_text = member_region.extract_text() or ""
                        lines = [l.strip() for l in member_text.splitlines() if l.strip()]

                        for i, line in enumerate(lines):
                            if "NPI" in line.upper():
                                if i + 1 < len(lines):
                                    next_line = lines[i + 1]
                                    m = re.match(
                                        r"([A-Z]+,\s*[A-Z]+(?:\s+[A-Z]+)?)",
                                        next_line, re.IGNORECASE
                                    )
                                    if m:
                                        patient_name = m.group(1).strip()
                                        break

                        if not patient_name:
                            for line in lines:
                                m = re.match(
                                    r"([A-Z]+,\s*[A-Z]+(?:\s+[A-Z]+)?)",
                                    line, re.IGNORECASE
                                )
                                if m:
                                    candidate = m.group(1).strip()
                                    if candidate.upper() not in {"IN NETWORK", "OUT OF NETWORK"}:
                                        patient_name = candidate
                                        break

                        print(f"  ✅ Patient Name: {patient_name}")

                    except Exception as e:
                        print(f"  ❌ Patient extraction error: {e}")

                    provider_name = extract_provider_name(
                        first_seg["page_obj"],
                        first_seg["start_y"],
                        first_seg["end_y"]
                    )
                    print(f" provider name: {provider_name}")

                    structured_table = {
                        "page":          first_seg["page_num"],
                        "table":         global_table_idx,
                        "patient_name":  patient_name,
                        "provider": provider_name,
                        "EOB_ID":        pdf_dir,
                        "denial_status": "Denied" if table_denied else "Not Denied",
                        "rows":          table.get("rows", []),
                        "column_totals": table.get("column_totals", {}),
                        "validation": {"status": is_valid, "errors": validation_errors},
                    }
                  

                    date_of_service = ""

                    if structured_table.get("rows"):
                        date_of_service = structured_table["rows"][0].get("dos", "")

                    for row in structured_table.get("rows", []):

                        row.pop("dos", None)


                    patient_data = {
                        "patient_name": structured_table.get("patient_name", ""),
                        "provider": structured_table.get("provider", ""),
                        "date_of_service": date_of_service,
                        "services": structured_table.get("rows", []),
                        "totals": structured_table.get("column_totals", {}),
                        "validation": {"status": is_valid, "errors": validation_errors},
                        "_expected_rows": expected_rows,
                        "_total_fields": total_fields,
                        "_model_confidence": table.get("_model_confidence", 0.0),
                    }                    

                    all_patients.append(patient_data)
                    confidence_results.append(patient_data)

            except json.JSONDecodeError as e:
                print(f"  JSON decode error: {e}")

            except Exception:
                traceback.print_exc()

    confidence_score = calculate_eob_confidence(confidence_results)


    final_output = [
        {
            "eob_id": pdf_dir,
            "file_name":pdf_full_name,
            "claim_status": "Denied" if claim_denied else "Not Denied",
            "payor": "UNITED HEALTHCARE",
            "confidence_score": confidence_score,
            "patients": all_patients
        }
    ]

    success_path, failed_path = save_split_output(
                                final_output,
                                company_name=company_name,
                                pdf_name=pdf_dir,
                                pdf_path=pdf_path,
                                cropped_dir=output_dir,
                            )
                        
    print(f"\n📁 Cropped images : {output_dir}")
    print(f"✅ Success json   : {success_path}")
    print(f"⚠  Failed json    : {failed_path}")
    return final_output


# =========================================================
# HELPER: Extract provider/patient name at specific y region
# =========================================================

def extract_provider_name_at_y(page, start_y, end_y):
    """
    Extracts the patient name from the member region of a segment.
    Looks for LASTNAME, FIRSTNAME pattern after NPI line.
    """
    try:
        member_region = page.crop((0, start_y + 70, 320, start_y + 170))
        member_text   = member_region.extract_text() or ""
        lines = [l.strip() for l in member_text.splitlines() if l.strip()]

        for i, line in enumerate(lines):
            if "NPI" in line.upper():
                if i + 1 < len(lines):
                    m = re.match(
                        r"([A-Z]+,\s*[A-Z]+(?:\s+[A-Z]+)?)",
                        lines[i + 1], re.IGNORECASE
                    )
                    if m:
                        return m.group(1).strip()

        for line in lines:
            m = re.match(
                r"([A-Z]+,\s*[A-Z]+(?:\s+[A-Z]+)?)",
                line, re.IGNORECASE
            )
            if m:
                candidate = m.group(1).strip()
                if candidate.upper() not in {"IN NETWORK", "OUT OF NETWORK"}:
                    return candidate

    except Exception:
        pass

    return ""


W0901 19:33:22.358000 3557392 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0901 19:33:22.373000 3557392 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [2]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/United_health/Uhc_comm_pdf/875521585.pdf")


Scanning Page 1
  Page 1 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 2
  Page 2 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 3
  Page 3 — denial keyword found: False
  provider_tops   = ['231.5']
  subtotal_bottoms= ['445.5']
  ✅ Provider@231.5 → SUB-TOTAL@445.5

Scanning Page 4
  Page 4 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 5
  Page 5 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 6
  Page 6 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 7
  Page 7 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 8
  Page 8 — denial keyword found: False
  provider_tops   = ['231.5']
  subtotal_bottoms= ['445.5']
  ✅ Provider@231.5 → SUB-TOTAL@445.5

Scanning Page 9
  Page 9 — denial keyword found: True
  provider_tops   = []
  su

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ amount_claimed              computed=861.0       | extracted=861.0       match
✅ amount_allowed              computed=468.0       | extracted=468.0       match
✅ deduct_applied              computed=0.0         | extracted=0.0         match
✅ other_ins                   computed=0.0         | extracted=0.0         match
✅ patient_resp                computed=57.0        | extracted=57.0        match
✅ amount_paid                 computed=411.0       | extracted=411.0       match
---------------------------------------------------------------------------
✅ [Table 1] Validation PASSED
  ✅ Patient Name: 
 provider name: ALEX RAMOS

--- Logical Table 2 (1 segment(s)) ---
  Cropped page 8: y=231.5→445.5
  Saved stitched image → EOB_OUTPUT/UHC_commercial/875521585/table_2.png
  🟢 NOT DENIED
Expected rows: 3


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ amount_claimed              computed=318.0       | extracted=318.0       match
✅ amount_allowed              computed=182.4       | extracted=182.4       match
✅ deduct_applied              computed=0.0         | extracted=0.0         match
✅ other_ins                   computed=0.0         | extracted=0.0         match
✅ patient_resp                computed=0.0         | extracted=0.0         match
✅ amount_paid                 computed=182.4       | extracted=182.4       match
---------------------------------------------------------------------------
✅ [Table 1] Validation PASSED
  ✅ Patient Name: JABLONSKI, MATTHEW
 provider name: DUC TANG

--- Logical Table 3 (1 segment(s)) ---
  Cropped page 13: y=231.5→417.1
  Saved stitched image → EOB_OUTPUT/UHC_commercial/875521585/table_3.png
  🟢 NOT DENIED
Expected rows: 2

🔍 Validation for [Table 1]
------------------------------------

[{'eob_id': '875521585',
  'file_name': '875521585.pdf',
  'claim_status': 'Not Denied',
  'payor': 'UNITED HEALTHCARE',
  'confidence_score': 99.0,
  'patients': [{'patient_name': '',
    'provider': 'ALEX RAMOS',
    'date_of_service': '04/23/26',
    'services': [{'procedure_code': 'D2391',
      'amount_claimed': '287.00',
      'amount_allowed': '156.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '19.00',
      'amount_paid': '137.00'},
     {'procedure_code': 'D2391',
      'amount_claimed': '287.00',
      'amount_allowed': '156.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '19.00',
      'amount_paid': '137.00'},
     {'procedure_code': 'D2391',
      'amount_claimed': '287.00',
      'amount_allowed': '156.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '19.00',
      'amount_paid': '137.00'}],
    'totals': {'amount_claimed': '$861.00',
     'amount_allowed': '

In [ ]:
crop_all_eob_tables(r"/home/cipl/users/OCR_Project/Solution_15_04/Jeeva/United_health/mistake_commercial_pdf_UH/denied_mistake/Pmt_EOP_824388491.pdf")


Scanning Page 1
  Page 1 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 2
  Page 2 — denial keyword found: False
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 3
  Page 3 — denial keyword found: False
  provider_tops   = ['231.5']
  subtotal_bottoms= ['530.5']
  ✅ Provider@231.5 → SUB-TOTAL@530.5

Scanning Page 4
  Page 4 — denial keyword found: True
  provider_tops   = []
  subtotal_bottoms= []

Scanning Page 5
  Page 5 — denial keyword found: True
  provider_tops   = []
  subtotal_bottoms= []

Total logical tables detected: 1

--- Logical Table 1 (1 segment(s)) ---


[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


  Cropped page 3: y=231.5→530.5
  Saved stitched image → EOB_OUTPUT/UHC_commercial/824388491/table_1.png
  🟢 NOT DENIED
Expected rows: 6


[transformers] Both `max_new_tokens` (=2048) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



🔍 Validation for [Table 1]
---------------------------------------------------------------------------
✅ amount_claimed              computed=1054.0      | extracted=1054.0      match
✅ amount_allowed              computed=649.0       | extracted=649.0       match
✅ deduct_applied              computed=0.0         | extracted=0.0         match
✅ other_ins                   computed=0.0         | extracted=0.0         match
✅ patient_resp                computed=237.0       | extracted=237.0       match
✅ amount_paid                 computed=412.0       | extracted=412.0       match
---------------------------------------------------------------------------
✅ [Table 1] Validation PASSED
  ✅ Patient Name: NGO, DUONG
 provider name: DUC TANG
✅ Success output + pdf + crops saved: EOB_output_success/UHC commercial/824388491

📁 Cropped images : EOB_OUTPUT/UHC_commercial/824388491
✅ Success json   : EOB_output_success/UHC commercial/824388491/824388491_output.json
⚠  Failed json    : None


[{'eob_id': '824388491',
  'claim_status': 'Not Denied',
  'payor': 'UNITED HEALTHCARE',
  'confidence_score': 99.0,
  'patients': [{'patient_name': 'NGO, DUONG',
    'provider': 'DUC TANG',
    'date_of_service': '01/30/26',
    'services': [{'procedure_code': 'D0274',
      'amount_claimed': '99.00',
      'amount_allowed': '57.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '0.00',
      'amount_paid': '57.00'},
     {'procedure_code': 'D0230',
      'amount_claimed': '37.00',
      'amount_allowed': '20.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '0.00',
      'amount_paid': '20.00'},
     {'procedure_code': 'D0150',
      'amount_claimed': '139.00',
      'amount_allowed': '73.00',
      'deduct_applied': '0.00',
      'other_ins': '0.00',
      'patient_resp': '0.00',
      'amount_paid': '73.00'},
     {'procedure_code': 'D0220',
      'amount_claimed': '45.00',
      'amount_allowed': '25.00',
     

: 